# PM2.5 Prediction — West Bengal

This notebook builds and compares regression models to predict PM2.5 concentration using air quality and meteorological data from West Bengal monitoring stations.

In [1]:
import numpy as np
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import lightgbm as lgb
import xgboost as xgb
import shap

## 1. Data Loading

In [2]:
path = r'C:\Users\palhr\OneDrive\Desktop\Pgdba Projects\RTSM\Data'

df_states = pd.read_csv(path + r'\stations_info.csv')
df_states.drop(columns=['agency', 'station_location', 'start_month'], inplace=True)

unique_states = df_states['state'].unique()
print("Available states:", unique_states)

Available states: ['Andhra Pradesh' 'Arunachal Pradesh' 'Assam' 'Bihar' 'Chhattisgarh'
 'Chandigarh' 'Delhi' 'Gujarat' 'Himachal Pradesh' 'Haryana' 'Jharkhand'
 'Jammu and Kashmir' 'Karnataka' 'Kerala' 'Maharashtra' 'Meghalaya'
 'Manipur' 'Madhya Pradesh' 'Mizoram' 'Nagaland' 'Odisha' 'Punjab'
 'Puducherry' 'Rajasthan' 'Sikkim' 'Telangana' 'Tamil Nadu' 'Tripura'
 'Uttarakhand' 'Uttar Pradesh' 'West Bengal']


In [3]:
def combine_state_df(state_name):
    DATASET_SRC = r"C:\Users\palhr\OneDrive\Desktop\Pgdba Projects\RTSM\Data"

    state_code = df_states[df_states['state'] == state_name]['file_name'].iloc[0][:2]
    state_files = glob.glob(f'{DATASET_SRC}\\{state_code}*.csv')
    print(f'Found {len(state_files)} files for {state_name}')

    combined_df = []
    for state_file in state_files:
        file_name = os.path.basename(state_file).replace('.csv', '').strip()
        match = df_states[df_states['file_name'].str.strip() == file_name]

        if match.empty:
            print(f'Skipping {file_name} — not found in station metadata')
            continue

        file_df = pd.read_csv(state_file)
        file_df['city'] = match['city'].values[0]
        file_df['city'] = file_df['city'].astype('string')
        combined_df.append(file_df)
        print(f'Loaded: {file_name}')

    if not combined_df:
        print("No files loaded — check df_states file_name column")
        return None

    return pd.concat(combined_df, ignore_index=True)


df = combine_state_df('West Bengal')
df.info()

Found 14 files for West Bengal
Loaded: WB001
Loaded: WB002
Loaded: WB003
Loaded: WB004
Loaded: WB005
Loaded: WB006
Loaded: WB007
Loaded: WB008
Loaded: WB009
Loaded: WB010
Loaded: WB011
Loaded: WB012
Loaded: WB013
Loaded: WB014
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 635495 entries, 0 to 635494
Data columns (total 36 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   From Date            635495 non-null  object 
 1   To Date              635495 non-null  object 
 2   PM2.5 (ug/m3)        463168 non-null  float64
 3   PM10 (ug/m3)         534764 non-null  float64
 4   NO (ug/m3)           515726 non-null  float64
 5   NO2 (ug/m3)          516386 non-null  float64
 6   NOx (ppb)            475868 non-null  float64
 7   NH3 (ug/m3)          414967 non-null  float64
 8   SO2 (ug/m3)          494849 non-null  float64
 9   CO (mg/m3)           525508 non-null  float64
 10  Ozone (ug/m3)        517328 non-null  float64

## 2. Data Preprocessing

In [4]:
df.drop(columns=['From Date', 'To Date', 'CH4 (ug/m3)',
                 'Eth-Benzene (ug/m3)', 'Xylene (ug/m3)'], inplace=True)

# Merge the two wind direction columns into one
df['wind_direction'] = df['WD (degree)'].fillna(df['WD (deg)'])

cols_to_keep = [
    'PM2.5 (ug/m3)',
    'NO2 (ug/m3)', 'NOx (ppb)', 'NH3 (ug/m3)', 'SO2 (ug/m3)',
    'CO (mg/m3)', 'Ozone (ug/m3)', 'WS (m/s)', 'wind_direction',
    'RH (%)', 'SR (W/mt2)', 'city'
]

df = df[cols_to_keep].copy()

df.columns = [
    'pm25', 'no2', 'nox', 'nh3', 'so2', 'co',
    'ozone', 'wind_speed', 'wind_direction',
    'humidity', 'solar_radiation', 'city'
]

print(df.shape)
df.head()

(635495, 12)


,pm25,no2,nox,nh3,so2,co,ozone,wind_speed,wind_direction,humidity,solar_radiation,city
0,NaN,21.62,23.27,NaN,20.51,0.62,11.94,NaN,NaN,NaN,16.16,Haldia
1,NaN,37.77,40.61,NaN,27.02,0.88,10.94,NaN,NaN,NaN,21.93,Haldia
2,NaN,17.08,18.94,NaN,16.14,0.63,17.05,NaN,NaN,NaN,15.74,Haldia
3,NaN,36.02,38.27,NaN,26.47,0.84,21.06,NaN,NaN,NaN,21.03,Haldia
4,NaN,37.58,39.73,NaN,24.74,0.96,20.36,NaN,NaN,NaN,21.41,Haldia


In [5]:
# Drop rows where PM2.5 is missing
before = len(df)
df.dropna(subset=['pm25'], inplace=True)
print(f"Rows dropped (PM2.5 missing): {before - len(df)}")
print(f"Rows remaining: {len(df)}")

# Fill remaining nulls with per-city median
numeric_cols = df.select_dtypes(include='float64').columns

for col in numeric_cols:
    df[col] = df.groupby('city')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Global median fallback
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Remove physically implausible PM2.5 readings
before = len(df)
df = df[(df['pm25'] >= 0) & (df['pm25'] <= 500)]
print(f"Outlier rows removed: {before - len(df)}")

# Encode city as numeric
le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])
print("City encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

Rows dropped (PM2.5 missing): 172327
Rows remaining: 463168
Outlier rows removed: 148
City encoding: {'Asansol': np.int64(0), 'Durgapur': np.int64(1), 'Haldia': np.int64(2), 'Howrah': np.int64(3), 'Kolkata': np.int64(4), 'Siliguri': np.int64(5)}


## 3. Feature Engineering

In [6]:
FEATURES_RAW = [
    'no2', 'nox', 'nh3', 'so2', 'co', 'ozone',
    'wind_speed', 'wind_direction', 'humidity',
    'solar_radiation', 'city_encoded'
]

log_candidates = ['no2', 'nox', 'nh3', 'so2', 'co', 'ozone', 'wind_speed', 'solar_radiation']

print("Skewness before log transform:")
for col in log_candidates:
    print(f"  {col:20s}: {df[col].skew():.2f}")

for col in log_candidates:
    df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

FEATURES_LOG = [
    'no2_log', 'nox_log', 'nh3_log', 'so2_log',
    'co_log', 'ozone_log', 'wind_speed_log', 'wind_direction',
    'humidity', 'solar_radiation_log', 'city_encoded'
]

print("\nSkewness after log transform:")
for col in log_candidates:
    print(f"  {col:20s}: {df[col+'_log'].skew():.2f}")

print(f"\nPM2.5 skewness: {df['pm25'].skew():.2f}")
df['pm25_log'] = np.log1p(df['pm25'])
TARGET_LOG = 'pm25_log'

X     = df[FEATURES_LOG]
y     = df[TARGET_LOG]
X_raw = df[FEATURES_RAW]
y_raw = df['pm25']

FEATURES = FEATURES_LOG

print(f"\nFinal dataset shape: {df.shape}  |  Nulls: {X.isnull().sum().sum()}")

Skewness before log transform:
  no2                 : 2.71
  nox                 : 4.12
  nh3                 : 5.03
  so2                 : 4.76
  co                  : 2.89
  ozone               : 1.81
  wind_speed          : 8.85
  solar_radiation     : 2.11

Skewness after log transform:
  no2                 : -0.05
  nox                 : 0.45
  nh3                 : -0.03
  so2                 : 0.19
  co                  : 1.28
  ozone               : -0.24
  wind_speed          : 1.15
  solar_radiation     : 0.40

PM2.5 skewness: 1.92

Final dataset shape: (463020, 22)  |  Nulls: 0


## 4. Pre-Model Diagnostics: Correlation Heatmap & VIF

In [7]:
# Heatmap — raw features
heatmap_cols = FEATURES_RAW + ['pm25']
corr = df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={"size": 9}, ax=ax)
ax.set_title("Pearson Correlation — Raw Features + PM2.5", fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("correlation_heatmap_raw.png", dpi=150, bbox_inches='tight')
plt.show()

target_corr = corr['pm25'].drop('pm25').sort_values(key=abs, ascending=False)
print("Correlation with PM2.5 (sorted by |r|):")
print(target_corr.round(3).to_string())

Correlation with PM2.5 (sorted by |r|):
co                 0.525
no2                0.490
nox                0.458
so2                0.364
nh3                0.240
humidity          -0.190
wind_direction     0.172
wind_speed        -0.106
solar_radiation   -0.057
city_encoded      -0.048
ozone             -0.037


In [8]:
# Heatmap — log-transformed features
heatmap_cols_log = FEATURES_LOG + ['pm25_log']
corr_log = df[heatmap_cols_log].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_log, dtype=bool))
sns.heatmap(corr_log, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={"size": 9}, ax=ax)
ax.set_title("Pearson Correlation — Log Features + Log PM2.5", fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("correlation_heatmap_log.png", dpi=150, bbox_inches='tight')
plt.show()

target_corr_log = corr_log['pm25_log'].drop('pm25_log').sort_values(key=abs, ascending=False)
print("Correlation with log(PM2.5) (sorted by |r|):")
print(target_corr_log.round(3).to_string())

Correlation with log(PM2.5) (sorted by |r|):
no2_log                0.519
nox_log                0.507
co_log                 0.504
so2_log                0.398
nh3_log                0.313
humidity              -0.282
wind_direction         0.203
wind_speed_log        -0.202
city_encoded          -0.087
ozone_log             -0.079
solar_radiation_log   -0.072


In [9]:
def compute_vif(X_df):
    X_arr = sm.add_constant(X_df.values, has_constant='add')
    vif_data = pd.DataFrame({
        'Feature': ['const'] + list(X_df.columns),
        'VIF': [variance_inflation_factor(X_arr, i) for i in range(X_arr.shape[1])]
    })
    return vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

scaler_r = StandardScaler()
scaler_l = StandardScaler()

vif_raw = compute_vif(pd.DataFrame(scaler_r.fit_transform(X_raw), columns=FEATURES_RAW))
vif_log = compute_vif(pd.DataFrame(scaler_l.fit_transform(X),     columns=FEATURES_LOG))

print("VIF — Raw features:")
print(vif_raw.round(2).to_string(index=False))

print("\nVIF — Log features:")
print(vif_log.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Variance Inflation Factor (VIF) — Multicollinearity Check", fontsize=13, fontweight='bold')

for ax, vif_df, title in zip(
        axes, [vif_raw, vif_log],
        ["Raw features (standardised)", "Log features (standardised)"]):
    colors = ['#e74c3c' if v > 10 else '#f39c12' if v > 5 else '#2ecc71' for v in vif_df['VIF']]
    bars = ax.barh(vif_df['Feature'], vif_df['VIF'], color=colors, edgecolor='white')
    ax.axvline(5,  color='#f39c12', linestyle='--', linewidth=1.2, label='VIF=5 (moderate)')
    ax.axvline(10, color='#e74c3c', linestyle='--', linewidth=1.2, label='VIF=10 (high)')
    for bar, v in zip(bars, vif_df['VIF']):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}', va='center', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("VIF")
    ax.legend(fontsize=8)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig("vif_chart.png", dpi=150, bbox_inches='tight')
plt.show()
print("VIF < 5: No multicollinearity | 5-10: Moderate | >10: High")

VIF — Raw features:
        Feature  VIF
            nox 2.83
            no2 2.69
       humidity 1.60
          ozone 1.59
             co 1.56
solar_radiation 1.26
            so2 1.19
            nh3 1.11
     wind_speed 1.04
 wind_direction 1.03
   city_encoded 1.03

VIF — Log features:
            Feature  VIF
            nox_log 3.84
            no2_log 3.61
          ozone_log 1.61
             co_log 1.50
           humidity 1.49
solar_radiation_log 1.33
            so2_log 1.23
            nh3_log 1.20
     wind_speed_log 1.13
     wind_direction 1.03
       city_encoded 1.03
VIF < 5: No multicollinearity | 5-10: Moderate | >10: High


## 5. Model Building

### Train / Test Split

In [10]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Fit scalers on train only
scaler_r = StandardScaler()
X_tr_sc  = pd.DataFrame(scaler_r.fit_transform(X_train_r), columns=FEATURES_RAW)
X_te_sc  = pd.DataFrame(scaler_r.transform(X_test_r),      columns=FEATURES_RAW)

scaler_l = StandardScaler()
X_tr_lsc = pd.DataFrame(scaler_l.fit_transform(X_train),   columns=FEATURES_LOG)
X_te_lsc = pd.DataFrame(scaler_l.transform(X_test),        columns=FEATURES_LOG)

print(f"Training set: {X_train.shape[0]} rows  |  Test set: {X_test.shape[0]} rows")

Training set: 370416 rows  |  Test set: 92604 rows


### 5.1 OLS Regression — Raw Features

In [11]:
X_tr_sc_c = sm.add_constant(X_tr_sc, has_constant='add')
X_te_sc_c = sm.add_constant(X_te_sc, has_constant='add')

ols_raw = sm.OLS(y_train_r.values, X_tr_sc_c).fit()
print(ols_raw.summary())

y_pred_ols_raw = ols_raw.predict(X_te_sc_c)

ols_raw_rmse = np.sqrt(mean_squared_error(y_test_r.values, y_pred_ols_raw))
ols_raw_mae  = mean_absolute_error(y_test_r.values, y_pred_ols_raw)
ols_raw_r2   = r2_score(y_test_r.values, y_pred_ols_raw)

print(f"\nOLS-Raw  RMSE : {ols_raw_rmse:.4f}")
print(f"OLS-Raw  MAE  : {ols_raw_mae:.4f}")
print(f"OLS-Raw  R2   : {ols_raw_r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_ols_raw, y_test_r.values - y_pred_ols_raw, alpha=0.2, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(title="OLS Raw — Residuals", xlabel="Predicted", ylabel="Residuals")

axes[1].scatter(y_test_r.values, y_pred_ols_raw, alpha=0.2, color='steelblue')
axes[1].plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--')
axes[1].set(title="OLS Raw — Actual vs Predicted", xlabel="Actual PM2.5", ylabel="Predicted PM2.5")

plt.tight_layout()
plt.savefig("ols_raw_plots.png", dpi=150)
plt.show()

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.427
Model:                            OLS   Adj. R-squared:                  0.427
Method:                 Least Squares   F-statistic:                 2.514e+04
Date:                Fri, 10 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:54:11   Log-Likelihood:            -1.8813e+06
No. Observations:              370416   AIC:                         3.763e+06
Df Residuals:                  370404   BIC:                         3.763e+06
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              57.6608      0.064    9

### 5.2 OLS Regression — Log-Transformed Features

In [12]:
X_tr_lsc_c = sm.add_constant(X_tr_lsc, has_constant='add')
X_te_lsc_c = sm.add_constant(X_te_lsc, has_constant='add')

ols_log = sm.OLS(y_train.values, X_tr_lsc_c).fit()
print(ols_log.summary())

y_pred_ols_log    = ols_log.predict(X_te_lsc_c)
y_pred_ols_log_bt = np.expm1(y_pred_ols_log)
y_test_bt         = np.expm1(y_test.values)

ols_log_rmse = np.sqrt(mean_squared_error(y_test_bt, y_pred_ols_log_bt))
ols_log_mae  = mean_absolute_error(y_test_bt, y_pred_ols_log_bt)
ols_log_r2   = r2_score(y_test_bt, y_pred_ols_log_bt)

print(f"\nOLS-Log  RMSE (original scale) : {ols_log_rmse:.4f}")
print(f"OLS-Log  MAE  (original scale) : {ols_log_mae:.4f}")
print(f"OLS-Log  R2   (original scale) : {ols_log_r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
resid = y_test.values - y_pred_ols_log
axes[0].scatter(y_pred_ols_log, resid, alpha=0.2, color='teal')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(title="OLS Log — Residuals (log scale)", xlabel="Predicted (log)", ylabel="Residuals")

axes[1].scatter(y_test_bt, y_pred_ols_log_bt, alpha=0.2, color='teal')
axes[1].plot([y_test_bt.min(), y_test_bt.max()], [y_test_bt.min(), y_test_bt.max()], 'r--')
axes[1].set(title="OLS Log — Actual vs Predicted (original scale)",
            xlabel="Actual PM2.5", ylabel="Predicted PM2.5")

plt.tight_layout()
plt.savefig("ols_log_plots.png", dpi=150)
plt.show()

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.483
Model:                            OLS   Adj. R-squared:                  0.483
Method:                 Least Squares   F-statistic:                 3.151e+04
Date:                Fri, 10 Apr 2026   Prob (F-statistic):               0.00
Time:                        23:54:14   Log-Likelihood:            -3.6009e+05
No. Observations:              370416   AIC:                         7.202e+05
Df Residuals:                  370404   BIC:                         7.203e+05
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   3.7089    

### 5.3 Polynomial Regression (Degree 2) with Ridge

Features with the strongest correlation to PM2.5 are selected for polynomial expansion. Standardisation is applied before expansion to prevent numerical issues, and Ridge regularisation is used since polynomial terms are inherently collinear.

In [13]:
poly_features = ['co', 'no2', 'nox', 'so2', 'nh3', 'humidity']

X_poly_base = df[poly_features].copy()
y_poly_raw  = df['pm25'].copy()

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly_base, y_poly_raw, test_size=0.2, random_state=42)

scaler_poly      = StandardScaler()
X_train_sc       = scaler_poly.fit_transform(X_train_poly)
X_test_sc        = scaler_poly.transform(X_test_poly)

poly             = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly_exp = poly.fit_transform(X_train_sc)
X_test_poly_exp  = poly.transform(X_test_sc)

print(f"Original features : {X_train_poly.shape[1]}")
print(f"After D2 expansion: {X_train_poly_exp.shape[1]}")

model_poly  = Ridge(alpha=1.0)
model_poly.fit(X_train_poly_exp, y_train_poly)
y_pred_poly = model_poly.predict(X_test_poly_exp)

poly_rmse = np.sqrt(mean_squared_error(y_test_poly, y_pred_poly))
poly_mae  = mean_absolute_error(y_test_poly, y_pred_poly)
poly_r2   = r2_score(y_test_poly, y_pred_poly)

def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

poly_adj_r2 = adjusted_r2(poly_r2, len(y_test_poly), X_test_poly_exp.shape[1])

X_train_sm  = sm.add_constant(X_train_poly_exp)
ols_poly_sm = sm.OLS(y_train_poly, X_train_sm).fit()
poly_aic    = ols_poly_sm.aic
poly_bic    = ols_poly_sm.bic

print(f"\nRMSE   : {poly_rmse:.4f}")
print(f"MAE    : {poly_mae:.4f}")
print(f"R2     : {poly_r2:.4f}")
print(f"Adj R2 : {poly_adj_r2:.4f}")
print(f"AIC    : {poly_aic:.0f}")
print(f"BIC    : {poly_bic:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_poly, y_test_poly - y_pred_poly, alpha=0.2, color='darkorange')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Polynomial Regression (D2) — Residuals')
axes[0].set_xlabel('Predicted PM2.5')
axes[0].set_ylabel('Residuals')

axes[1].scatter(y_test_poly, y_pred_poly, alpha=0.2, color='darkorange')
axes[1].plot([y_test_poly.min(), y_test_poly.max()],
             [y_test_poly.min(), y_test_poly.max()], 'r--')
axes[1].set_title('Polynomial Regression (D2) — Actual vs Predicted')
axes[1].set_xlabel('Actual PM2.5')
axes[1].set_ylabel('Predicted PM2.5')

plt.tight_layout()
plt.savefig('polynomial_plots.png', dpi=150, bbox_inches='tight')
plt.show()

Original features : 6
After D2 expansion: 27

RMSE   : 37.5205
MAE    : 25.7742
R2     : 0.4696
Adj R2 : 0.4695
AIC    : 3733685
BIC    : 3733988


### 5.4 LightGBM

In [14]:
lgb_params = {
    "objective": "regression", "metric": "rmse",
    "learning_rate": 0.05, "num_leaves": 63,
    "feature_fraction": 0.8, "bagging_fraction": 0.8,
    "bagging_freq": 5, "min_child_samples": 20,
    "lambda_l1": 0.1, "lambda_l2": 0.1, "verbose": -1,
}

lgb_model = lgb.train(
    lgb_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=1000,
    valid_sets=[lgb.Dataset(X_test, label=y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_pred_lgb    = lgb_model.predict(X_test)
y_pred_lgb_bt = np.expm1(y_pred_lgb)

lgb_rmse = np.sqrt(mean_squared_error(y_test_bt, y_pred_lgb_bt))
lgb_mae  = mean_absolute_error(y_test_bt, y_pred_lgb_bt)
lgb_r2   = r2_score(y_test_bt, y_pred_lgb_bt)

print(f"LightGBM RMSE : {lgb_rmse:.4f}")
print(f"LightGBM MAE  : {lgb_mae:.4f}")
print(f"LightGBM R2   : {lgb_r2:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test_bt, y_pred_lgb_bt, alpha=0.2, color='#4CAF50')
ax.plot([y_test_bt.min(), y_test_bt.max()], [y_test_bt.min(), y_test_bt.max()], 'r--')
ax.set(title="LightGBM — Actual vs Predicted", xlabel="Actual PM2.5", ylabel="Predicted PM2.5")
plt.tight_layout()
plt.savefig("lgb_avp.png", dpi=150)
plt.show()

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.516885
[200]	valid_0's rmse: 0.495019
[300]	valid_0's rmse: 0.48244
[400]	valid_0's rmse: 0.474492
[500]	valid_0's rmse: 0.468404
[600]	valid_0's rmse: 0.463451
[700]	valid_0's rmse: 0.459194
[800]	valid_0's rmse: 0.455475
[900]	valid_0's rmse: 0.452307
[1000]	valid_0's rmse: 0.449858
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 0.449858
LightGBM RMSE : 28.3948
LightGBM MAE  : 17.6817
LightGBM R2   : 0.6963


### 5.5 XGBoost

In [15]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    early_stopping_rounds=50, eval_metric="rmse",
    random_state=42, verbosity=0
)

xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)], verbose=100)

y_pred_xgb    = xgb_model.predict(X_test)
y_pred_xgb_bt = np.expm1(y_pred_xgb)

xgb_rmse = np.sqrt(mean_squared_error(y_test_bt, y_pred_xgb_bt))
xgb_mae  = mean_absolute_error(y_test_bt, y_pred_xgb_bt)
xgb_r2   = r2_score(y_test_bt, y_pred_xgb_bt)

print(f"XGBoost RMSE : {xgb_rmse:.4f}")
print(f"XGBoost MAE  : {xgb_mae:.4f}")
print(f"XGBoost R2   : {xgb_r2:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test_bt, y_pred_xgb_bt, alpha=0.2, color='#FF5722')
ax.plot([y_test_bt.min(), y_test_bt.max()], [y_test_bt.min(), y_test_bt.max()], 'r--')
ax.set(title="XGBoost — Actual vs Predicted", xlabel="Actual PM2.5", ylabel="Predicted PM2.5")
plt.tight_layout()
plt.savefig("xgb_avp.png", dpi=150)
plt.show()

[0]	validation_0-rmse:3.17089
[100]	validation_0-rmse:0.53143
[200]	validation_0-rmse:0.51383
[300]	validation_0-rmse:0.50035
[400]	validation_0-rmse:0.49100
[500]	validation_0-rmse:0.48466
[600]	validation_0-rmse:0.48000
[700]	validation_0-rmse:0.47581
[800]	validation_0-rmse:0.47218
[900]	validation_0-rmse:0.46943
[999]	validation_0-rmse:0.46679
XGBoost RMSE : 29.5271
XGBoost MAE  : 18.4258
XGBoost R2   : 0.6715


## 6. Model Comparison

In [16]:
def adj_r2(r2, n, k):
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)

n_test = X_test.shape[0]

results = pd.DataFrame({
    "Model"  : ["OLS-Raw", "OLS-Log", "Poly-Reg (D2)", "LightGBM", "XGBoost"],
    "RMSE"   : [ols_raw_rmse, ols_log_rmse, poly_rmse,  lgb_rmse,  xgb_rmse],
    "MAE"    : [ols_raw_mae,  ols_log_mae,  poly_mae,   lgb_mae,   xgb_mae],
    "R2"     : [ols_raw_r2,   ols_log_r2,   poly_r2,    lgb_r2,    xgb_r2],
    "Adj R2" : [
        adj_r2(ols_raw_r2, X_test_r.shape[0],   len(FEATURES_RAW)),
        adj_r2(ols_log_r2, X_test.shape[0],     len(FEATURES_LOG)),
        adj_r2(poly_r2,    X_test_poly.shape[0], X_test_poly_exp.shape[1]),
        adj_r2(lgb_r2,     n_test,              len(FEATURES)),
        adj_r2(xgb_r2,     n_test,              len(FEATURES)),
    ],
}).sort_values("R2", ascending=False).reset_index(drop=True)

results["Rank"] = results.index + 1

print("=" * 65)
print("        FINAL MODEL COMPARISON (original PM2.5 scale)")
print("=" * 65)
print(results.round(4).to_string(index=False))
print("=" * 65)

        FINAL MODEL COMPARISON (original PM2.5 scale)
        Model    RMSE     MAE     R2  Adj R2  Rank
     LightGBM 28.3948 17.6817 0.6963  0.6962     1
      XGBoost 29.5271 18.4258 0.6715  0.6715     2
Poly-Reg (D2) 37.5205 25.7742 0.4696  0.4695     3
      OLS-Raw 39.0069 26.6573 0.4268  0.4267     4
      OLS-Log 43.4743 26.8396 0.2880  0.2879     5


In [17]:
model_colors = {
    "OLS-Raw"       : "#9E9E9E",
    "OLS-Log"       : "#2196F3",
    "Poly-Reg (D2)" : "#FF9800",
    "LightGBM"      : "#4CAF50",
    "XGBoost"       : "#FF5722",
}

metrics = ["R2", "Adj R2", "RMSE", "MAE"]
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.suptitle("Model Comparison — All Metrics (original PM2.5 scale)",
             fontsize=14, fontweight='bold')

for ax, m in zip(axes, metrics):
    models = results["Model"].tolist()
    vals   = results[m].tolist()
    bars   = ax.bar(models, vals,
                    color=[model_colors[mo] for mo in models],
                    edgecolor='white', linewidth=1.2, width=0.5)

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{v:.3f}", ha='center', va='bottom',
                fontsize=8, fontweight='bold')

    best = vals.index(max(vals) if m in ["R2", "Adj R2"] else min(vals))
    ax.patches[best].set_edgecolor('gold')
    ax.patches[best].set_linewidth(3)

    ax.set_title(m, fontweight='bold')
    ax.set_ylabel(m)
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=8)

    patches = [plt.Rectangle((0, 0), 1, 1, color=model_colors[mo], label=mo) for mo in models]
    ax.legend(handles=patches, fontsize=7)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved model_comparison.png")

Saved model_comparison.png


## 7. SHAP Analysis — LightGBM Feature Importance

In [18]:
shap_sample = X_test.sample(n=25000, random_state=42)

explainer   = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(shap_sample)

# Summary dot plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, shap_sample,
                  feature_names=FEATURES, show=False)
plt.title('SHAP Summary — LightGBM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved shap_summary.png')

# Bar plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, shap_sample,
                  feature_names=FEATURES, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — LightGBM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved shap_bar.png')

Saved shap_summary.png
Saved shap_bar.png
